# Integrity Scoring · End-to-End Walkthrough

演示从一份**虚构公司档案**(EXAMPLE_CORP)出发,完整跑一遍诚信评分:

1. 加载 promises(从公司 YAML 档案)
2. 由你(署名研究者)给出 verdicts(notebook 内手动定义)
3. 调用 `compute_integrity_score`
4. 解释 breakdown
5. 演示 `severity_tag` 放大机制

> ⚠️ 本 notebook **只用虚构公司**(EXAMPLE_CORP)。任何真实公司的 verdict 必须由真实研究者署名后,通过仓库 PR 流程进入,**不应在 notebook 中匿名生成**。

In [ ]:
import sys
from datetime import date
from pathlib import Path

# notebook 在 integrity_framework/examples/ 下,加 repo 根到 sys.path
REPO_ROOT = Path.cwd().resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from integrity_framework.src import (
    Promise,
    Verdict,
    compute_integrity_score,
    load_promises_from_company_yaml,
)

EXAMPLE_PATH = REPO_ROOT / "companies" / "_examples" / "example-company.yaml"
print("loading", EXAMPLE_PATH.relative_to(REPO_ROOT))

## 步骤 1 · 从档案加载 promises

`load_promises_from_company_yaml` 读 YAML 中 `integrity_tracking.tracked_promises[]`,返回一组 `Promise` 对象。

In [ ]:
promises = load_promises_from_company_yaml(EXAMPLE_PATH)
for p in promises:
    print(f"[{p.id}] {p.promise_zh[:60]}...")
    print(f"    verification_type={p.verification_type}, due_by={p.due_by}, ticker={p.subject_ticker}")
print(f"\n total: {len(promises)} promises")

## 步骤 2 · 你(署名研究者)给出 verdicts

**关键约束**:每个 `Verdict` 必须带 `judged_by`(不接受 `anonymous`)和 `reasoning`(空字符串会被拒绝)。下面是模拟 4 种典型情况:

In [ ]:
verdicts = [
    Verdict(
        promise_id="fy24-revenue-25pct",
        outcome="broken",
        reasoning="示例:FY24 实际营收 +18%,低于指引 7pp",
        judged_at=date(2025, 4, 30),
        judged_by="@example-analyst",
        evidence_urls=("https://example.com/fy24-results",),
    ),
    Verdict(
        promise_id="agent-product-launch",
        outcome="partial",
        reasoning="示例:产品在 Q2 而非承诺的 Q1 上线,功能缩减",
        judged_at=date(2025, 7, 10),
        judged_by="@example-analyst",
        evidence_urls=("https://example.com/q2-launch-news",),
    ),
    Verdict(
        promise_id="fy25-cost-reduction",
        outcome="pending",
        reasoning="示例:还未到验证窗口(Q4 财报披露后再 verdict)",
        judged_at=date(2026, 1, 15),
        judged_by="@example-analyst",
    ),
    Verdict(
        promise_id="fy24-overseas-expansion",
        outcome="fulfilled",
        reasoning="示例:H2 已在印尼 + 越南上线,符合公开承诺",
        judged_at=date(2025, 12, 1),
        judged_by="@example-analyst",
    ),
]
for v in verdicts:
    print(f"  {v.promise_id} → {v.outcome.upper()} (judged by {v.judged_by})")

## 步骤 3 · 算分

`as_of` 是评分时间锚(影响时间衰减)。这里用一个固定日期让结果可复现。

In [ ]:
score = compute_integrity_score(verdicts, as_of=date(2026, 4, 24))
print(f"分数: {score.score} / 100")
print(f"算法版本: {score.methodology_version}")
print(f"参与计算: {score.counted_verdicts}")
print(f"跳过(pending/unverifiable): {score.skipped_verdicts}")

## 步骤 4 · 看 breakdown

**没有 breakdown 的分数等于黑盒**——读分时必须看具体每条 verdict 的 `final_delta`。

In [ ]:
print(f"{'promise_id':<30} {'outcome':<12} {'raw':>6} {'decay':>6} {'sev':>6} {'final':>7}  notes")
print("-" * 100)
for b in score.breakdown:
    print(
        f"{b.promise_id:<30} {b.outcome:<12} {b.raw_delta:>6.1f} "
        f"{b.decay_multiplier:>6.2f} {b.severity_bonus:>6.1f} {b.final_delta:>7.2f}  {b.notes}"
    )

## 步骤 5 · severity_tag 放大演示

如果同一个 broken verdict 带了 `financial_misconduct` 严重性标签,分数会被额外扣 30 分。

In [ ]:
amplified = Verdict(
    promise_id="fy24-revenue-25pct",
    outcome="broken",
    reasoning="示例:FY24 营收虚增,后被审计调整,触发 SEC enforcement",
    judged_at=date(2025, 4, 30),
    judged_by="@example-analyst",
    severity_tag="financial_misconduct",
)
amplified_score = compute_integrity_score([amplified], as_of=date(2026, 4, 24))
print(f"无放大: 100 - 15 = 85")
print(f"有 financial_misconduct 放大: {amplified_score.score} (= 100 - 15 - 30)")

## 步骤 6 · 序列化为 JSON(下游消费)

`IntegrityScore.to_dict()` 返回 plain dict,直接 `json.dumps` 可以喂给 web 层 / API。

In [ ]:
import json
print(json.dumps(score.to_dict(), ensure_ascii=False, indent=2))

## 收尾

- 所有数据都是**虚构**(EXAMPLE_CORP + 示例 verdicts)
- 真实公司的 verdict 必须由真实研究者署名,通过 PR 流程进入仓库
- 算法权重(`-5 / -15 / -30`)是 v0.1.0 placeholder,等积累 50+ 真实 verdict 后会回测调整
- 完整方法论见 [`../METHODOLOGY.md`](../METHODOLOGY.md);边界讨论见 [`../docs/`](../docs/)

更多问题 → GitHub `methodology_discussion` issue。